<table>
<tr>                                                                                   
     <th>
         <div style='padding:15px;color:#030aa7;font-size:240%;text-align: center;font-style: italic;font-weight: bold;font-family: Georgia, serif'><a href="https://www.kaggle.com/datasets/pitt/contagious-diseases">Project Tycho-Contagious Diseases</a></div>
     </th>
     <th><img src=https://raw.githubusercontent.com/rbizoi/IntelligenceEnDonneesDeSante/refs/heads/main/images/tycho_cd.jpg width="96"></th>
 </tr>
</table>

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Introduction</div></b>
## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Import libriries </div></b>

In [1]:
import pandas as pd, numpy as np, seaborn as sns, warnings, os
from datetime import datetime as dt
from matplotlib import pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patheffects as path_effects

import plotly.express as px
import plotly.graph_objs as go

font1 = fm.FontProperties(size=20)
font2 = fm.FontProperties(size=24)

warnings.filterwarnings(action="ignore")

if int(str(sns.__version__).split('.')[1]) > 8 : 
    plt.style.use('seaborn-v0_8-darkgrid')
else:
    plt.style.use('seaborn-darkgrid')
sns.set(font_scale=3)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)  
pd.set_option('display.max_rows', None)

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Lecture des données</div></b>

In [2]:
!ls -al "../donnees/Project Tycho-Contagious Diseases"

total 22976
drwxrwxrwx  3 razvan razvan    4096 nov.  24 18:52  .
drwxrwxrwx 35 razvan razvan    4096 nov.  18 12:04  ..
-rwxrwxrwx  1 razvan razvan 2771718 oct.   7  2021 'archive(23).zip'
-rwxrwxrwx  1 razvan razvan 3463284 sept. 20  2019  hepatitis.csv
drwxrwxr-x  2 razvan razvan    4096 nov.  24 18:52  .ipynb_checkpoints
-rwxrwxrwx  1 razvan razvan 5053296 sept. 20  2019  measles.csv
-rwxrwxrwx  1 razvan razvan 2184265 sept. 20  2019  mumps.csv
-rwxrwxrwx  1 razvan razvan 3928121 sept. 20  2019  pertussis.csv
-rwxrwxrwx  1 razvan razvan 2570114 sept. 20  2019  polio.csv
-rwxrwxrwx  1 razvan razvan     345 oct.   7  2021 'Project Tycho Contagious Diseases Kaggle.URL'
-rwxrwxrwx  1 razvan razvan 1758930 sept. 20  2019  rubella.csv
-rwxrwxrwx  1 razvan razvan 1730116 sept. 20  2019  smallpox.csv


In [3]:
repertoire="../donnees/Project Tycho-Contagious Diseases"
fichiers = {}
maladies = {'hepatitis':'hépatite','measles':'rougeole','mumps':'oreillons','pertussis':'coqueluche','polio':'poliomyélite','rubella':'rubéole','smallpox':'variole'}
for fichier in os.listdir(repertoire): 
    if fichier[-4:] == '.csv' :
        fichiers[maladies[fichier[:-4]]] = pd.read_csv(os.path.join(repertoire, fichier)).set_index(['week', 'state', 'state_name'])

for maladie in fichiers:
    fichiers[maladie].rename(columns={'disease':maladie, 'cases':f'{maladie}_c', 'incidence_per_capita':f'{maladie}_ic'},inplace=True)
    print(f'{maladie} ',end='\t')
    print(f'{fichiers[maladie].shape[0]}    ',end='\t')
    print(f'{fichiers[maladie].reset_index()[['week', 'state', 'state_name']].drop_duplicates().shape[0]}     ',end='\t')
    print(f'{fichiers[maladie].shape[0] - fichiers[maladie].reset_index()[['week', 'state', 'state_name']].drop_duplicates().shape[0]}')

oreillons 	69754    	69754     	0
hépatite 	90839    	90839     	0
variole 	50916    	50916     	0
rubéole 	53205    	53205     	0
poliomyélite 	81531    	81518     	13
coqueluche 	109072    	109072     	0
rougeole 	145167    	145165     	2


## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Poliomyélite</div></b>

In [4]:
fichiers['poliomyélite'].reset_index(inplace=True)
fichiers['poliomyélite']['n'] = fichiers['poliomyélite'].groupby(['week','state','state_name']).week.transform('count') 
fichiers['poliomyélite'] = fichiers['poliomyélite'][(fichiers['poliomyélite'].n != 2)|(fichiers['poliomyélite'].poliomyélite_c != r'\N')]  
fichiers['poliomyélite'].drop(columns='n',inplace=True)
fichiers['poliomyélite'].set_index(['week', 'state', 'state_name'], inplace=True)
fichiers['poliomyélite'].shape[0]

81518

In [5]:
fichiers['poliomyélite'].poliomyélite_c[fichiers['poliomyélite'].poliomyélite_c == r'\N'] = 0
fichiers['poliomyélite'].poliomyélite_c = fichiers['poliomyélite'].poliomyélite_c.astype('int32')

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Rougeole</div></b>

In [6]:
fichiers['rougeole'].reset_index(inplace=True)
fichiers['rougeole']['n'] = fichiers['rougeole'].groupby(['week','state','state_name']).week.transform('count') 
fichiers['rougeole'][fichiers['rougeole'].n > 1]

,week,state,state_name,rougeole,rougeole_c,rougeole_ic,n
140443,199551,MA,MASSACHUSETTS,MEASLES,1,0.02,2
140444,199551,MA,MASSACHUSETTS,MEASLES,1,0.02,2
140456,199551,WA,WASHINGTON,MEASLES,1,0.02,2
140457,199551,WA,WASHINGTON,MEASLES,1,0.02,2


In [7]:
fichiers['rougeole'] = fichiers['rougeole'].drop_duplicates()
fichiers['rougeole'].drop(columns='n',inplace=True)
fichiers['rougeole'].set_index(['week', 'state', 'state_name'], inplace=True)

In [8]:
fichiers['rougeole'].shape[0]

145165

## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Jointure</div></b>

In [9]:
cles = list(fichiers.keys())
nom = cles[0]
donnees = fichiers[nom] 

cle_avant = cles[0]
for cle in cles[1:]:
    donnees = donnees.join(fichiers[cle], how='outer') # ,lsuffix = f'_{cle_avant}', rsuffix = f'_{cle}'
    cle_avant = cle

In [10]:
for maladie in fichiers:
    print(f'{maladie} \t fichier {fichiers[maladie].shape[0]}   \t donnees {sum(~donnees[maladie].isna())}')

oreillons 	 fichier 69754   	 donnees 69754
hépatite 	 fichier 90839   	 donnees 90839
variole 	 fichier 50916   	 donnees 50916
rubéole 	 fichier 53205   	 donnees 53205
poliomyélite 	 fichier 81518   	 donnees 81518
coqueluche 	 fichier 109072   	 donnees 109072
rougeole 	 fichier 145165   	 donnees 145165


In [11]:
donnees.shape

(209823, 21)

In [12]:
donnees.reset_index(inplace=True)

In [13]:
donnees['année'] = donnees.week.apply(lambda x : str(x)[:4])

In [14]:
donnees.head()

,week,state,state_name,oreillons,oreillons_c,oreillons_ic,hépatite,hépatite_c,hépatite_ic,variole,variole_c,variole_ic,rubéole,rubéole_c,rubéole_ic,poliomyélite,poliomyélite_c,poliomyélite_ic,coqueluche,coqueluche_c,coqueluche_ic,rougeole,rougeole_c,rougeole_ic,année
0,192801,AL,ALABAMA,NaN,NaN,NaN,NaN,NaN,NaN,SMALLPOX,1.0,0.04,NaN,NaN,NaN,POLIO,0.0,0.00,NaN,NaN,NaN,MEASLES,97.0,3.67,1928
1,192801,AR,ARKANSAS,NaN,NaN,NaN,NaN,NaN,NaN,SMALLPOX,7.0,0.38,NaN,NaN,NaN,POLIO,0.0,0.00,NaN,NaN,NaN,MEASLES,76.0,4.11,1928
2,192801,AZ,ARIZONA,NaN,NaN,NaN,NaN,NaN,NaN,SMALLPOX,0.0,0.00,NaN,NaN,NaN,POLIO,0.0,0.00,NaN,NaN,NaN,MEASLES,8.0,1.90,1928
3,192801,CA,CALIFORNIA,NaN,NaN,NaN,NaN,NaN,NaN,SMALLPOX,18.0,0.34,NaN,NaN,NaN,POLIO,9.0,0.17,NaN,NaN,NaN,MEASLES,74.0,1.38,1928
4,192801,CO,COLORADO,NaN,NaN,NaN,NaN,NaN,NaN,SMALLPOX,31.0,3.06,NaN,NaN,NaN,POLIO,4.0,0.39,NaN,NaN,NaN,MEASLES,85.0,8.38,1928


In [15]:
donnees[['oreillons_c', 'hépatite_c', 'variole_c', 'rubéole_c', 'poliomyélite_c', 'coqueluche_c', 'rougeole_c', 'année']].groupby('année').sum()

,oreillons_c,hépatite_c,variole_c,rubéole_c,poliomyélite_c,coqueluche_c,rougeole_c
année,,,,,,,
1928,0.0,0.0,36470.0,0.0,4756.0,0.0,483337.0
1929,0.0,0.0,38389.0,0.0,2746.0,0.0,339061.0
1930,0.0,0.0,45728.0,0.0,8964.0,0.0,384597.0
1931,0.0,0.0,28708.0,0.0,15743.0,0.0,438435.0
1932,0.0,0.0,10740.0,0.0,3829.0,0.0,390114.0
1933,0.0,0.0,6164.0,0.0,4927.0,0.0,380394.0
1934,0.0,0.0,5139.0,0.0,7274.0,0.0,727096.0
1935,0.0,0.0,7489.0,0.0,10733.0,0.0,721002.0
1936,0.0,0.0,7262.0,0.0,4326.0,0.0,280942.0
